# 04 动态规划：策略迭代与价值迭代

**前提**：已知环境模型 $P(s'|s,a)$ 和 $R(s,a,s')$。

**算法**：
1. **策略迭代** = 策略评估 + 策略改进 交替
2. **价值迭代** = 一直对 V 做 Bellman 最优 backup

**环境**：4×4 GridWorld
```
[T  1  2  3 ]
[4  5  6  7 ]
[8  9  10 11]
[12 13 14 T ]
```
T = terminal 终止格；每步 reward = -1；目标是最快到达终止格。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

GRID_SIZE = 4
N_STATES = GRID_SIZE * GRID_SIZE
ACTIONS = ['UP', 'DOWN', 'LEFT', 'RIGHT']
ACTION_DELTAS = {'UP': (-1, 0), 'DOWN': (1, 0), 'LEFT': (0, -1), 'RIGHT': (0, 1)}
TERMINAL_STATES = [0, 15]
GAMMA = 1.0

def state_to_pos(s):
    return s // GRID_SIZE, s % GRID_SIZE

def pos_to_state(r, c):
    return r * GRID_SIZE + c

def step(s, a):
    if s in TERMINAL_STATES:
        return s, 0.0
    r, c = state_to_pos(s)
    dr, dc = ACTION_DELTAS[a]
    nr, nc = max(0, min(GRID_SIZE-1, r+dr)), max(0, min(GRID_SIZE-1, c+dc))
    return pos_to_state(nr, nc), -1.0

## 1. 策略评估 (Policy Evaluation)

给定策略 $\pi$，迭代式求 $V^\pi$：

$$V_{k+1}(s) = \sum_a \pi(a|s) \sum_{s'} P(s'|s,a)[R + \gamma V_k(s')]$$

本环境转移确定，简化为：

$$V_{k+1}(s) = \sum_a \pi(a|s) [R(s,a) + \gamma V_k(s')]$$

In [ ]:
def policy_evaluation(policy, theta=1e-4):
    """policy: shape (N_STATES, len(ACTIONS)) -> 概率"""
    V = np.zeros(N_STATES)
    while True:
        delta = 0.0
        for s in range(N_STATES):
            if s in TERMINAL_STATES:
                continue
            v_old = V[s]
            v_new = 0.0
            for a_idx, a in enumerate(ACTIONS):
                s_next, r = step(s, a)
                v_new += policy[s, a_idx] * (r + GAMMA * V[s_next])
            V[s] = v_new
            delta = max(delta, abs(v_old - v_new))
        if delta < theta:
            break
    return V

uniform_policy = np.ones((N_STATES, len(ACTIONS))) / len(ACTIONS)
V_random = policy_evaluation(uniform_policy)
print('随机策略下的 V:')
print(V_random.reshape(GRID_SIZE, GRID_SIZE).round(1))

## 2. 策略改进 (Policy Improvement)

$$\pi'(s) = \arg\max_a \sum_{s'} P(s'|s,a)[R + \gamma V^\pi(s')]$$

In [ ]:
def policy_improvement(V):
    new_policy = np.zeros((N_STATES, len(ACTIONS)))
    for s in range(N_STATES):
        if s in TERMINAL_STATES:
            new_policy[s] = 1.0 / len(ACTIONS)
            continue
        q_values = []
        for a in ACTIONS:
            s_next, r = step(s, a)
            q_values.append(r + GAMMA * V[s_next])
        best = np.argmax(q_values)
        new_policy[s, best] = 1.0
    return new_policy

def policy_iteration():
    policy = np.ones((N_STATES, len(ACTIONS))) / len(ACTIONS)
    iterations = 0
    while True:
        V = policy_evaluation(policy)
        new_policy = policy_improvement(V)
        iterations += 1
        if np.array_equal(new_policy, policy):
            break
        policy = new_policy
    print(f'策略迭代收敛于 {iterations} 轮')
    return policy, V

pi_star, V_star = policy_iteration()
print('\n最优 V:')
print(V_star.reshape(GRID_SIZE, GRID_SIZE).round(1))

## 3. 价值迭代 (Value Iteration)

直接对 V 做 Bellman 最优 backup，省去中间策略评估：

$$V_{k+1}(s) = \max_a \sum_{s'} P(s'|s,a)[R + \gamma V_k(s')]$$

In [ ]:
def value_iteration(theta=1e-4):
    V = np.zeros(N_STATES)
    iterations = 0
    while True:
        delta = 0.0
        for s in range(N_STATES):
            if s in TERMINAL_STATES:
                continue
            v_old = V[s]
            q_values = []
            for a in ACTIONS:
                s_next, r = step(s, a)
                q_values.append(r + GAMMA * V[s_next])
            V[s] = max(q_values)
            delta = max(delta, abs(v_old - V[s]))
        iterations += 1
        if delta < theta:
            break
    pi = policy_improvement(V)
    print(f'价值迭代收敛于 {iterations} 轮')
    return pi, V

pi_vi, V_vi = value_iteration()
print('\n价值迭代得到的 V:')
print(V_vi.reshape(GRID_SIZE, GRID_SIZE).round(1))

## 4. 可视化最优策略

用箭头展示在每个格子的最优动作。

In [ ]:
def plot_policy(policy, V, title):
    fig, ax = plt.subplots(figsize=(5, 5))
    arrow_map = {'UP': (0, 0.3), 'DOWN': (0, -0.3), 'LEFT': (-0.3, 0), 'RIGHT': (0.3, 0)}
    grid = V.reshape(GRID_SIZE, GRID_SIZE)
    ax.imshow(grid, cmap='RdYlGn')
    for s in range(N_STATES):
        r, c = state_to_pos(s)
        if s in TERMINAL_STATES:
            ax.text(c, r, 'T', ha='center', va='center', fontsize=20, fontweight='bold')
        else:
            best_a = ACTIONS[np.argmax(policy[s])]
            dx, dy = arrow_map[best_a]
            ax.arrow(c, r, dx, -dy, head_width=0.15, color='black')
        ax.text(c, r+0.4, f'{V[s]:.1f}', ha='center', fontsize=8)
    ax.set_title(title)
    ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()

plot_policy(pi_vi, V_vi, 'Optimal Policy (Value Iteration)')

## 5. 总结与思考

| 方法 | 每轮代价 | 收敛轮数 | 适用 |
|-----|---------|---------|-----|
| 策略迭代 | 高（含完整评估） | 少 | 状态/动作小 |
| 价值迭代 | 低 | 多 | 通用 |

**关键限制**：DP 要求已知 $P, R$，且状态数有限。真实 RL 问题里我们一般 **不知道转移**，所以下一步要进入 **MC** 和 **TD**。

**思考**：如果 GridWorld 里某些格子有 30% 概率"打滑"（不动），如何修改 `step` 函数？怎么影响最优策略？